# InfoQuant-01 — Intrication quantique : états mixtes, Werner, PPT, distillabilité

Premier carnet de la sous-série **InformationQuantique**. Il introduit six notions centrales de la théorie de l'information quantique, chacune **avec un exemple calculé** (numpy, des matrices plutôt que des phrases) :

1. les **états mixtes** et leurs purifications ;
2. l'**intrication** d'un état pur composé (décomposition de Schmidt) — et sa différence explicite avec l'« intrication » au sens de la sous-série [Chaînes](../Chaines/Chaines-01-Axiome-Intrication-Constante-Structure-Fine-Python.ipynb) ;
3. les **états de Werner** $\rho_\alpha$, famille à un paramètre qui sépare séparable / PPT / distillable ;
4. le **critère PPT** (Peres–Horodecki) : la transposition partielle comme test de séparabilité ;
5. la **distillabilité** : à partir de copies d'un état mixte, retrouver du singulet par LOCC ;
6. les **canaux entanglement-breaking** : critère de Choi, forme de Holevo, indice EB.

Chaque section suit le même patron : un cadre bref, le calcul, puis la lecture du résultat. Aucune bibliothèque de calcul quantique n'est utilisée — tout est reconstruit avec `numpy` pour que chaque affirmation du texte soit un nombre affiché par une cellule.

## 0. Deux « intrications » à ne pas confondre

Le dépôt emploie le mot *intrication* dans deux sens distincts, et ce carnet est celui du second :

| | Sous-série **Chaînes** (Chain Theory) | Ce carnet (**information quantique**) |
|---|---|---|
| Objet | une **paire** kat de coefficients $C$, ses états réduits $\rho = CC^\dagger$ aux deux bouts | un **état composé** $\rho$ sur $\mathcal{H}_A \otimes \mathcal{H}_B$, matrice densité $d^2 \times d^2$ |
| « Intrication » | structure de la paire : support commun $m$, rang de $C$, écart à l'égalité des degrés | **enchevêtrement quantique** : $\rho$ n'est pas décomposable en mélange de produits $\rho'_A \otimes \rho''_B$ |
| Test calculé | rang et spectre de l'état réduit | décomposition de Schmidt (§2), critère PPT (§4) |
| Témoin opérationnel | transmission d'une constante sur le lien | violation d'une inégalité de Bell (CHSH — prouvé formellement dans la série Lean du dépôt : `SymbolicAI/Lean/Lean-13b-CHSH-Tsirelson-Native.ipynb` et `Lean-13c-CHSH-Landau-Saturation.ipynb`, que ce carnet cite sans les réimplémenter) |

La définition opératoire de ce carnet (Peres, 1996) : un état **séparable** s'écrit

$$\rho = \sum_A w_A\, \rho'_A \otimes \rho''_A, \qquad w_A \geq 0, \quad \sum_A w_A = 1,$$

et tout état qui n'admet pas une telle décomposition est **intriqué**. Toute la suite consiste à tester cette décomposabilité par des moyens calculables.

In [1]:
import numpy as np

rng = np.random.default_rng(42)
print("numpy", np.__version__)


def tp_b(rho, d1, d2):
    """Transpose partielle sur le facteur B : sigma_{m,mu,n,nu} = rho_{m,nu,n,mu}.
    Convention de Peres 1996 (eq. 3) : on transpose les indices d'un seul sous-systeme."""
    R = rho.reshape(d1, d2, d1, d2)
    return R.transpose(0, 3, 2, 1).reshape(d1 * d2, d1 * d2)


def ket(*etats):
    """Produit tensoriel de kets locaux (vecteurs numpy 1-D)."""
    v = np.array([1.0])
    for e in etats:
        v = np.kron(v, np.asarray(e, dtype=complex))
    return v


def val_propres(rho):
    """Valeurs propres reelles d'une matrice hermitienne, triees croissant."""
    return np.sort(np.real(np.linalg.eigvalsh(rho)))


# Le singulet de deux qubits, acteur central de tout le carnet
e0, e1 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
psi_moins = (ket(e0, e1) - ket(e1, e0)) / np.sqrt(2)
print("|psi-> =", psi_moins.real)

numpy 2.4.2
|psi-> = [ 0.          0.70710678 -0.70710678  0.        ]


## 1. États mixtes et purifications

Un **état pur** est un vecteur normalisé $|\psi\rangle$ ; sa matrice densité $\rho = |\psi\rangle\langle\psi|$ est de rang 1. Un **état mixte** modélise l'ignorance : le système est dans l'état pur $\rho_A$ avec probabilité $w_A$, et seule la statistique

$$\rho = \sum_A w_A\, \rho_A, \qquad w_A \geq 0,\ \sum_A w_A = 1$$

est connue. La **pureté** $\operatorname{tr}(\rho^2) \in [1/d, 1]$ vaut $1$ si et seulement si l'état est pur — c'est le discriminant calculable entre mélange et superposition.

Une **purification** de $\rho$ (sur $\mathcal{H}_A \otimes \mathcal{H}_R$) est un état pur $|\Psi\rangle$ dont l'état réduit redonne $\rho$ : $\operatorname{tr}_R |\Psi\rangle\langle\Psi| = \rho$. Toute matrice densité en admet une de dimension au plus $d \times d$, construite sur sa décomposition spectrale :

$$|\Psi\rangle = \sum_k \sqrt{w_k}\ |a_k\rangle \otimes |k\rangle.$$

In [2]:
# Un etat mixte d'un qubit, melange explicite de trois etats purs
poids = np.array([0.5, 0.3, 0.2])
etats = [e0, e1, (e0 + e1) / np.sqrt(2)]

rho = sum(w * np.outer(e, e.conj()) for w, e in zip(poids, etats))
print("trace(rho)        =", round(float(np.trace(rho).real), 12))
print("valeurs propres   =", val_propres(rho).round(6), " (toutes >= 0 : rho est un etat physique)")
purete = float(np.trace(rho @ rho).real)
print("purete tr(rho^2)  =", round(purete, 6), " (strictement < 1 : etat mixte)")

# Controle : un etat pur donne bien purete 1
rho_pur = np.outer(psi_moins, psi_moins.conj())
print("purete du singulet =", round(float(np.trace(rho_pur @ rho_pur).real), 12))

trace(rho)        = 1.0
valeurs propres   = [0.358579 0.641421]  (toutes >= 0 : rho est un etat physique)
purete tr(rho^2)  = 0.54  (strictement < 1 : etat mixte)
purete du singulet = 1.0


### Lecture du résultat

Le mélange $(0.5, 0.3, 0.2)$ des trois états purs donne une pureté de $0.54$ : strictement inférieure à $1$ alors même que chaque composante est pure. C'est la signature que le mélange est de l'**ignorance statistique** (des poids classiques), pas une superposition quantique — le singulet, lui, reste à pureté exactement $1$. Les valeurs propres non négatives et la trace $1$ vérifient que $\rho$ est bien une matrice densité.

In [3]:
# Purification de rho : |Psi> = somme_k sqrt(w_k) |a_k>|k> sur A (2 dims) x ancille R (2 dims)
w, V = np.linalg.eigh(rho)
d = rho.shape[0]
base_ancille = np.eye(d)

psi_pur = np.zeros(d * d, dtype=complex)
for k in range(d):
    psi_pur = psi_pur + np.sqrt(max(w[k], 0.0)) * ket(V[:, k], base_ancille[:, k])

# Verification 1 : tr_R(|Psi><Psi|) reconstruit exactement rho
M = np.outer(psi_pur, psi_pur.conj()).reshape(d, d, d, d)   # indices (a, r, a', r')
rho_reduit = np.einsum('abcb->ac', M)                        # somme sur l'ancille
print("tr_R(|Psi><Psi|) == rho :", bool(np.allclose(rho_reduit, rho)))

# Verification 2 : la purete de l'etat reduit retombe sur celle de rho
print("tr(rho_reduit^2) =", round(float(np.trace(rho_reduit @ rho_reduit).real), 6),
      " vs tr(rho^2) =", round(purete, 6))

tr_R(|Psi><Psi|) == rho : True
tr(rho_reduit^2) = 0.54  vs tr(rho^2) = 0.54


### Lecture du résultat

La purification construite sur le spectre de $\rho$ passe les deux tests : la trace partielle sur l'ancille reconstruit $\rho$ **exactement** (`allclose` vrai), et la pureté de l'état réduit retombe sur celle de $\rho$. C'est le pont conceptuel que la suite exploite : *tout état mixte peut être vu comme l'état réduit d'un état pur sur un espace plus grand* — et la question de l'intrication de $\rho$ devient celle de l'intrication de cette purification entre $A$ et $R$.

## 2. Intrication d'un état pur : la décomposition de Schmidt

Pour un état pur $|\psi\rangle$ de $\mathcal{H}_A \otimes \mathcal{H}_B$ ($\dim A = d_1$, $\dim B = d_2$), on écrit les coefficients dans la base produit : $|\psi\rangle = \sum_{ij} C_{ij}\, |i\rangle|j\rangle$. La **décomposition de Schmidt** est la valeur singulière de cette matrice :

$$|\psi\rangle = \sum_k s_k\, |u_k\rangle |v_k\rangle, \qquad s_k \geq 0,$$

où les $s_k$ (coefficients de Schmidt) sont les valeurs singulières de $C$. Le **rang de Schmidt** compte les $s_k > 0$, et

$$|\psi\rangle \text{ est produit} \iff \text{rang de Schmidt} = 1 \iff \operatorname{tr}(\rho_A^2) = 1.$$

Les carrés $s_k^2$ sont exactement les valeurs propres de l'état réduit $\rho_A$ (et de $\rho_B$) — c'est ce spectre partagé qui porte l'intrication.

In [4]:
def coefficients_schmidt(psi, d1, d2):
    """Coefficients de Schmidt = valeurs singulieres de la matrice C des coefficients."""
    return np.linalg.svd(np.asarray(psi).reshape(d1, d2), compute_uv=False)


def etat_reduit_A(psi, d1, d2):
    """Etat reduit rho_A = tr_B |psi><psi| ; son spectre est le carre des s_k."""
    M = np.outer(psi, psi.conj()).reshape(d1, d2, d1, d2)
    return np.einsum('abcb->ac', M)


# Trois etats purs : produit, maximalement intrique, aleatoire (2 x 4)
phi_prod = ket(e0, e1)
phi_plus = (ket(e0, e0) + ket(e1, e1)) / np.sqrt(2)
v = rng.normal(size=8) + 1j * rng.normal(size=8)
psi_rand = v / np.linalg.norm(v)

for nom, psi, (d1, d2) in [
    ("|01>                         ", phi_prod, (2, 2)),
    ("(|00>+|11>)/sqrt(2)          ", phi_plus, (2, 2)),
    ("etat pur aleatoire (2 x 4)   ", psi_rand, (2, 4)),
]:
    s = coefficients_schmidt(psi, d1, d2)
    vp = val_propres(etat_reduit_A(psi, d1, d2))
    rang = int(np.sum(s > 1e-9))
    print(f"{nom} rang Schmidt = {rang}  s = {np.sort(s)[::-1].round(6)}")
    print(f"{'':30} spectre rho_A = {vp[::-1].round(6)}  (= s^2)")

|01>                          rang Schmidt = 1  s = [1. 0.]
                               spectre rho_A = [1. 0.]  (= s^2)
(|00>+|11>)/sqrt(2)           rang Schmidt = 2  s = [0.707107 0.707107]
                               spectre rho_A = [0.5 0.5]  (= s^2)
etat pur aleatoire (2 x 4)    rang Schmidt = 2  s = [0.864907 0.501932]
                               spectre rho_A = [0.748064 0.251936]  (= s^2)


### Lecture du résultat

- $|01\rangle$ : un seul coefficient de Schmidt non nul (rang 1), $\rho_A$ de spectre $\{1, 0\}$ — un état **produit**, l'intrication est nulle.
- $(|00\rangle + |11\rangle)/\sqrt{2}$ : deux coefficients égaux à $1/\sqrt{2}$, $\rho_A = I/2$ de spectre $\{0.5, 0.5\}$ — **maximalement intriqué** : chaque sous-système pris seul est complètement aléatoire alors que le tout est pur.
- L'état aléatoire $2 \times 4$ : rang 2, coefficients inégaux — intriqué (rang > 1) mais pas maximal.

L'intrication d'un état pur est donc entièrement lisible dans le spectre de l'état réduit : **plat = maximalement intriqué, un pic en $1$ = produit**. C'est le sens quantique du mot, distinct de la structure de paire de la sous-série Chaînes (§0). La violation CHSH qui en est le témoin opérationnel est démontrée formellement dans les carnets Lean-13b/13c du dépôt, que ce carnet se contente de citer.

## 3. Les états de Werner : une famille à un paramètre

Pour un état **pur**, l'intrication se lit sur le spectre de l'état réduit (§2). Pour un état **mixte**, le mélange brouille ce spectre : l'état réduit d'un mélange de produits est encore diagonal plat, sans rien dire de la séparabilité. La famille de Werner — introduite par R. F. Werner en 1989 (référence et paramétrisation : revue Horodecki *et al.* 2009, éq. 71) — est le laboratoire minimal de cette question : un seul paramètre, et déjà toutes les subtilités.

À deux qubits, version de Peres (1996, éq. 7) : un mélange de singulet et d'état maximalement mélangé,

$$\rho(p) = p\, |\psi^-\rangle\langle\psi^-| + (1-p)\, \frac{I}{4},$$

En dimension générale $d \otimes d$, la définition de la revue (éq. 71) melange les projecteurs $P_\pm = (I \pm V)/2$ sur les sous-espaces symétrique et antisymétrique ($V$ l'opérateur de retournement, $V|ij\rangle = |ji\rangle$) :

$$W(\lambda) = (1-\lambda)\, \frac{2}{d(d+1)}\, P_+ + \lambda\, \frac{2}{d(d-1)}\, P_-,$$

où $\lambda = \operatorname{tr}(P_- W)$ est le **poids antisymétrique**. Les deux paramétrisations coïncident à $d=2$ (où $P_- = |\psi^-\rangle\langle\psi^-|$) via $\lambda = (1+3p)/4$.

In [5]:
def werner_2q(p):
    """Werner a deux qubits (parametrisation de Peres, eq. 7) :
    rho(p) = p |psi-><psi-| + (1-p) I/4 ; p = fraction de singulet."""
    S = np.outer(psi_moins, psi_moins.conj())
    return p * S + (1 - p) * np.eye(4) / 4


# Correspondance des parametrisations : lambda = <psi-|rho(p)|psi-> = (1+3p)/4
print("p      lambda=(1+3p)/4   vp(rho^G_B) minimale")
for p in [0.0, 1/9, 1/3, 0.5, 1.0]:
    rho = werner_2q(p)
    lam = float(np.real(psi_moins.conj() @ (rho @ psi_moins)))
    print(f"{p:.4f}  {lam:.6f}         {val_propres(tp_b(rho, 2, 2))[0]:+.6f}")

print()
print("seuil Peres p = 1/3  <=>  seuil poids antisymetrique lambda = 1/2")

p      lambda=(1+3p)/4   vp(rho^G_B) minimale
0.0000  0.250000         +0.250000
0.1111  0.333333         +0.166667
0.3333  0.500000         +0.000000
0.5000  0.625000         -0.125000
1.0000  1.000000         -0.500000

seuil Peres p = 1/3  <=>  seuil poids antisymetrique lambda = 1/2


### Lecture du résultat

La valeur propre minimale de $\rho^{\Gamma_B}$ vaut $(1-3p)/4$ — elle passe par zéro à $p = 1/3$ exactement (c'est le calcul de Peres 1996, éq. 7-8, revérifié ici chiffre par chiffre : les trois autres valeurs propres, $(1+p)/4$, restent positives). La correspondance des paramétrisations donne $\lambda = (1+3p)/4$, donc le seuil $p = 1/3$ se dit aussi $\lambda = 1/2$. À ce stade le signe de cette valeur propre n'est qu'un **constat spectral** ; la section suivante lui donne son sens : c'est le critère PPT.

In [6]:
def flip(d):
    """Operateur de retournement V|ij> = |ji> sur C^d (x) C^d."""
    V = np.zeros((d * d, d * d))
    for i in range(d):
        for j in range(d):
            V[i * d + j, j * d + i] = 1.0
    return V


def werner_d(lmbda, d):
    """Werner d (x) d, revue Horodecki et al. 2009 eq. 71 :
    W(l) = (1-l) 2/(d(d+1)) P_+ + l 2/(d(d-1)) P_- ; l = poids antisymetrique."""
    V = flip(d)
    P_plus = (np.eye(d * d) + V) / 2
    P_moins = (np.eye(d * d) - V) / 2
    return (1 - lmbda) * 2 / (d * (d + 1)) * P_plus + lmbda * 2 / (d * (d - 1)) * P_moins


# Coherence : en d = 2, W(lambda) coincide avec rho(p) pour p = (4 lambda - 1)/3
print("d=2 : W(0.25) == rho((4*0.25-1)/3)  :", bool(np.allclose(werner_d(0.25, 2), werner_2q(0.0))))
print("d=2 : W(0.75) == rho((4*0.75-1)/3)  :", bool(np.allclose(werner_d(0.75, 2), werner_2q(2.0 / 3.0))))

# En dimension d = 3 : le meme seuil spectral lambda = 1/2
d = 3
print(f"\nd={d} : lambda   vp(W) minimale   vp(W^G_B) minimale")
for lmbda in [0.4, 0.5, 0.6, 0.8]:
    W = werner_d(lmbda, d)
    print(f"     {lmbda:.2f}      {val_propres(W)[0]:+.6f}        {val_propres(tp_b(W, d, d))[0]:+.6f}")

d=2 : W(0.25) == rho((4*0.25-1)/3)  : True
d=2 : W(0.75) == rho((4*0.75-1)/3)  : True

d=3 : lambda   vp(W) minimale   vp(W^G_B) minimale
     0.40      +0.100000        +0.066667
     0.50      +0.083333        +0.000000
     0.60      +0.066667        -0.066667
     0.80      +0.033333        -0.200000


### Lecture du résultat

Les deux définitions coïncident en $d = 2$ (`allclose` vrai aux deux extrémités du segment), ce qui valide le code de la version générale. En $d = 3$, le tableau montre deux faits :

- les valeurs propres de $W(\lambda)$ elles-mêmes sont **toujours positives** (c'est un état physique pour tout $\lambda$) ;
- la valeur propre minimale de la transposée partielle $W^{\Gamma_B}$ s'annule à $\lambda = 1/2$ et devient négative au-delà — **le même seuil qu'à deux qubits**.

La revue 2009 (éq. 71 et discussion) établit que pour les états de Werner, séparable $\iff$ PPT $\iff$ $\lambda \leq 1/2$, en **toute** dimension. Le calcul ci-dessus en vérifie le volet spectral en $d = 3$.

## 4. Le critère PPT (Peres–Horodecki)

L'idée de Peres (1996) : si $\rho = \sum_A w_A\, \rho'_A \otimes \rho''_A$ est séparable, alors sa transposée partielle

$$\sigma_{m\mu,n\nu} \equiv \rho_{n\mu,m\nu} \qquad \text{(éq. 3 de Peres ; ici } \rho^{\Gamma_B}\text{)}$$

est encore un mélange $\sum_A w_A\, (\rho'_A)^T \otimes \rho''_A$ de matrices positives — donc $\rho^{\Gamma_B} \geq 0$. Un état dont la transposée partielle a une valeur propre négative est **nécessairement intriqué** : c'est le critère **PPT** (*positive partial transpose*). Peres calcule pour le Werner à deux qubits les valeurs propres $3 \times (1+x)/4$ et $(1-3x)/4$, et donne un second exemple frappant : un mélange de singulet et d'une paire maximalement polarisée, intriqué pour **tout** $x > 0$.

La réciproque est vraie en dimensions $2 \times 2$ et $2 \times 3$ (Horodecki, Horodecki et Horodecki, 1996) : là, PPT $\iff$ séparable. En dimensions supérieures elle est **fausse** — il existe des états PPT intriqués (l'« enchevêtrement lié », *bound entanglement*, revue 2009), et des états non-PPT dont le statut exige des critères plus fins (§5).

In [7]:
S = np.outer(psi_moins, psi_moins.conj())

# Exemple 1 de Peres (eq. 7) : verification exacte du spectre de sigma
print("Exemple 1 (eq. 7) : rho = x S + (1-x) I/4")
for x in [0.2, 1/3, 0.5, 0.9]:
    rho = x * S + (1 - x) * np.eye(4) / 4
    vp = val_propres(tp_b(rho, 2, 2))
    attendu = np.sort([(1 - 3 * x) / 4] + [(1 + x) / 4] * 3)
    print(f"  x={x:.4f} : vp(sigma) = {vp.round(6)}  attendu = {attendu.round(6)}  ok={np.allclose(vp, attendu)}")

# Exemple 3 de Peres (eq. 15) : singulet + paire maximalement polarisee |00>
print("\nExemple 3 (eq. 15) : rho = x S + (1-x) |00><00|")
ket00 = ket(e0, e0)
for x in [0.01, 0.05, 0.5]:
    rho = x * S + (1 - x) * np.outer(ket00, ket00)
    print(f"  x={x:.2f} : vp(sigma) minimale = {val_propres(tp_b(rho, 2, 2))[0]:+.6f}"
          f"   (Bell reste verifiee jusqu'a x ~ 0.8 : PPT voit l'intrication bien avant)")

Exemple 1 (eq. 7) : rho = x S + (1-x) I/4
  x=0.2000 : vp(sigma) = [0.1 0.3 0.3 0.3]  attendu = [0.1 0.3 0.3 0.3]  ok=True
  x=0.3333 : vp(sigma) = [0.       0.333333 0.333333 0.333333]  attendu = [0.       0.333333 0.333333 0.333333]  ok=True
  x=0.5000 : vp(sigma) = [-0.125  0.375  0.375  0.375]  attendu = [-0.125  0.375  0.375  0.375]  ok=True
  x=0.9000 : vp(sigma) = [-0.425  0.475  0.475  0.475]  attendu = [-0.425  0.475  0.475  0.475]  ok=True

Exemple 3 (eq. 15) : rho = x S + (1-x) |00><00|
  x=0.01 : vp(sigma) minimale = -0.000025   (Bell reste verifiee jusqu'a x ~ 0.8 : PPT voit l'intrication bien avant)
  x=0.05 : vp(sigma) minimale = -0.000657   (Bell reste verifiee jusqu'a x ~ 0.8 : PPT voit l'intrication bien avant)
  x=0.50 : vp(sigma) minimale = -0.103553   (Bell reste verifiee jusqu'a x ~ 0.8 : PPT voit l'intrication bien avant)


### Lecture du résultat

- **Exemple 1** : le spectre calculé de $\sigma$ coïncide au dernier chiffre avec les valeurs $3 \times (1+x)/4$ et $(1-3x)/4$ publiées par Peres (colonne `attendu`) — la transposition partielle devient détectrice d'intrication dès $x > 1/3$, là où l'inégalité de Bell ne réagit qu'à partir de $1/\sqrt{2}$ et l'inégalité d'entropie à $1/\sqrt{3}$.
- **Exemple 3**, le plus frappant : le mélange singulet + $|00\rangle$ a une valeur propre de $\sigma$ **négative pour tout $x > 0$**, même $x = 0.01$ — un état à $99\%$ de produit local reste intriqué. Peres note que pour cet état, l'inégalité de Bell reste satisfaite jusqu'à $x \approx 0.8$ : PPT est spectaculairement plus fin.

Le théorème de Horodecki *et al.* (1996) donne la réciproque en $2\times 2$ et $2\times 3$ : là, PPT $\iff$ séparable, et le test ci-dessus est **décisif**. En dimension supérieure, le spectre négatif de $\rho^{\Gamma_B}$ reste un certificat d'intrication, mais son absence ne garantit plus la séparabilité — c'est la porte de l'enchevêtrement lié et de la question ouverte du §5.

## 5. Distillabilité : combien de copies pour un singulet ?

La **distillation** (Bennett *et al.*, 1996) : à partir d'un grand nombre de copies d'un état mixte $\rho$, deux parties ne communiquant que classiquement (LOCC) cherchent à extraire des singulets presque purs. Un état est **distillable** si c'est possible pour un $n$ assez grand.

Le critère calculable (revue Horodecki *et al.* 2009, note 49) : $\rho$ est distillable si et seulement s'il existe $n$ et un vecteur $|\psi\rangle$ de **rang de Schmidt 2** tels que

$$\langle\psi|\,(\rho^{\otimes n})^{\Gamma}\,|\psi\rangle < 0.$$

Deux conséquences immédiates, que nous calculons :

1. **PPT $\Rightarrow$ non distillable** — la valeur attendue reste positive pour tout $|\psi\rangle$ ;
2. à deux qubits, le témoin à une copie est $|\Phi^+\rangle = (|00\rangle+|11\rangle)/\sqrt{2}$ — le vecteur propre négatif de $\rho(p)^{\Gamma}$, qui n'est **pas** l'état du mélange — avec $\langle\Phi^+|\rho(p)^{\Gamma}|\Phi^+\rangle = (1-3p)/4 < 0 \iff p > 1/3$ : **tout état de Werner à deux qubits non-PPT est distillable avec une seule copie**.

En dimension $d \geq 3$, un **trou** apparaît : Dür *et al.* (2000) montrent que pour les états de Werner, l'état est séparable pour $\lambda \leq 1/2$ mais **1-distillable seulement pour** $\lambda > 3(d-1)/(2(2d-1))$ — seuil qui tend vers $3/4$ quand $d \to \infty$. Entre $1/2$ et ce seuil, l'état est non-PPT (donc intriqué) et pourtant aucun protocole à une copie ne fonctionne : c'est la zone où vit la **question ouverte n° 2** de la revue 2009 — *tout état NPT est-il distillable ?* (ramenée à : tout état de Werner NPT est-il distillable, Horodecki–Horodecki 1999).

In [8]:
# (a) Deux qubits : le temoin de Schmidt rang 2 n'est PAS le singulet -- c'est |Phi+>,
#     l'eigenvecteur negatif de rho(p)^G dans la sous-base {00, 11}.
phi_plus2 = (ket(e0, e0) + ket(e1, e1)) / np.sqrt(2)
print("(a) d=2 : <psi-|rho^G|psi-> (echoue)   <Phi+|rho^G|Phi+> (temoin)   (1-3p)/4")
for p in [0.25, 1/3, 0.5, 1.0]:
    rho_g = tp_b(werner_2q(p), 2, 2)
    v_sing = float(np.real(psi_moins.conj() @ (rho_g @ psi_moins)))
    v_phi = float(np.real(phi_plus2.conj() @ (rho_g @ phi_plus2)))
    print(f"    p={p:.4f} : {v_sing:+.6f}                {v_phi:+.6f}              {(1 - 3*p)/4:+.6f}")


# (b) d=3 : minimum sur la famille de Schmidt rang 2 |psi> = cos(t)|ii> + sin(t)|jj>.
#     Par la forme fermee W^G = (a+b) I + d(a-b) |Psi_max><Psi_max| (voir la lecture),
#     minimiser sur les vecteurs de rang 2 revient a maximiser le recouvrement avec
#     |Psi_max>, atteint en t = pi/4 : valeur (3 - 5*lambda)/12, nulle en lambda = 3/5.
def min_temoin_schmidt2(Wg, d, n_theta=400):
    meilleur = np.inf
    for i in range(d):
        for j in range(i + 1, d):
            for th in np.linspace(0.0, np.pi / 2, n_theta):
                psi = np.cos(th) * ket(np.eye(d)[i], np.eye(d)[i]) \
                    + np.sin(th) * ket(np.eye(d)[j], np.eye(d)[j])
                v = float(np.real(psi.conj() @ (Wg @ psi)))
                meilleur = min(meilleur, v)
    return meilleur


d = 3
seuil_dur = 3 * (d - 1) / (2 * (2 * d - 1))
print(f"\n(b) d={d} : seuil 1-distillabilite de Duer et al. = 3(d-1)/(2(2d-1)) = {seuil_dur:.6f}")
print("    lambda   vp(W^G) min   min Schmidt-2 (numerique)   forme fermee (3-5l)/12")
for lmbda in [0.45, 0.55, 0.6, 0.65, 0.7]:
    W = werner_d(lmbda, d)
    vp_min = val_propres(tp_b(W, d, d))[0]
    m = min_temoin_schmidt2(tp_b(W, d, d), d)
    attendu = (3 - 5 * lmbda) / 12
    verdict = "temoin < 0 : 1-distillable" if m < -1e-9 else "pas de temoin a une copie"
    print(f"    {lmbda:.2f}    {vp_min:+.6f}        {m:+.6f}              {attendu:+.6f}   {verdict}")

(a) d=2 : <psi-|rho^G|psi-> (echoue)   <Phi+|rho^G|Phi+> (temoin)   (1-3p)/4
    p=0.2500 : +0.312500                +0.062500              +0.062500
    p=0.3333 : +0.333333                +0.000000              +0.000000
    p=0.5000 : +0.375000                -0.125000              -0.125000
    p=1.0000 : +0.500000                -0.500000              -0.500000

(b) d=3 : seuil 1-distillabilite de Duer et al. = 3(d-1)/(2(2d-1)) = 0.600000
    lambda   vp(W^G) min   min Schmidt-2 (numerique)   forme fermee (3-5l)/12


    0.45    +0.033333        +0.062500              +0.062500   pas de temoin a une copie


    0.55    -0.033333        +0.020834              +0.020833   pas de temoin a une copie
    0.60    -0.066667        +0.000001              +0.000000   pas de temoin a une copie


    0.65    -0.100000        -0.020833              -0.020833   temoin < 0 : 1-distillable


    0.70    -0.133333        -0.041666              -0.041667   temoin < 0 : 1-distillable


### Lecture du résultat

- **(a)** La valeur d'attente sur le singulet reste **positive** pour tout $p$ — un piège instructif : l'état qui compose le mélange n'est pas le témoin de sa distillabilité. Le bon témoin est $|\Phi^+\rangle$, vecteur propre négatif de $\rho(p)^{\Gamma}$ : la colonne du milieu réalise exactement $(1-3p)/4$, négatif dès $p > 1/3$. Comme PPT $\iff$ séparable en $2 \times 2$, *tout état de Werner à deux qubits intriqué est distillable avec une seule copie*.
- **(b)** En $d = 3$, le tableau sépare trois régimes : $\lambda \leq 1/2$ (PPT, colonne « vp minimale » positive), puis le trou $1/2 < \lambda \leq 3/5$ où $W^{\Gamma}$ a une valeur propre négative (état **intriqué**) mais le minimum sur les vecteurs de Schmidt rang 2 reste positif — la 1-distillabilité échoue — puis $\lambda > 3/5$ où le témoin apparaît. Le minimum numérique épouse la forme fermée $(3-5\lambda)/12$, nulle **exactement** en $\lambda = 3/5$ : c'est le seuil $3(d-1)/(2(2d-1))$ de Dür *et al.*, ici reproduit par calcul direct.

D'où vient cette forme fermée ? La transposition partielle de l'opérateur de retournement est $V^{\Gamma} = d\,|\Psi_{\max}\rangle\langle\Psi_{\max}|$, donc

$$W(\lambda)^{\Gamma} = (a+b)\, I + d\,(a-b)\, |\Psi_{\max}\rangle\langle\Psi_{\max}|,$$

et minimiser $\langle\psi|W^{\Gamma}|\psi\rangle$ sur les vecteurs de rang de Schmidt 2 revient à **maximiser le recouvrement** $|\langle\Psi_{\max}|\psi\rangle|^2$ — plafonné à $2/d$ (vecteurs $(|ii\rangle+|jj\rangle)/\sqrt{2}$). C'est toute la subtilité du passage $d = 2 \to d = 3$ : le vecteur pleinement intriqué $|\Psi_{\max}\rangle$, témoin idéal de la non-PPT-itude, a un rang de Schmidt $d$ et devient **inadmissible** comme témoin de distillabilité.

In [9]:
# Recursion de Duer et al. 2000 : pour l'etat depolarise (par LOCC) de N copies de W(lambda),
# le poids antisymetrique suit lambda_N = (1 - (1 - 2 lambda)^N) / 2
lam = 0.55
print(f"lambda_1 = {lam} (dans le trou : NPT mais pas 1-distillable en d=3)")
print("N     lambda_N")
for N in range(1, 8):
    lamN = (1 - (1 - 2 * lam) ** N) / 2
    print(f"{N}     {lamN:.6f}")

print("\nPropriete (Duer et al.) : lambda_N <= lambda_1 pour tout N --")
print("le poids antisymetrique DIMINUE quand on prend des copies :")
print("les etats de Werner du trou ne s'approfondissent pas en NPT, ils oscillent autour de 1/2.")

lambda_1 = 0.55 (dans le trou : NPT mais pas 1-distillable en d=3)
N     lambda_N
1     0.550000
2     0.495000
3     0.500500
4     0.499950
5     0.500005
6     0.499999
7     0.500000

Propriete (Duer et al.) : lambda_N <= lambda_1 pour tout N --
le poids antisymetrique DIMINUE quand on prend des copies :
les etats de Werner du trou ne s'approfondissent pas en NPT, ils oscillent autour de 1/2.


### Lecture du résultat

Parti de $\lambda_1 = 0.55$ — à peine au-dessus du seuil de séparabilité — le poids antisymétrique des copies successives oscille en s'amortissant autour de $1/2$ (0.55, 0.495, 0.5005, …), conformément à la propriété $\lambda_N \leq \lambda_1$ démontrée par Dür *et al.* : **prendre des copies n'aggrave pas l'intrication détectable**, il l'érode au contraire. Les protocoles de distillation ne peuvent donc pas compter sur un effet de cumul simple, et c'est précisément pourquoi la question « tout état NPT est-il distillable ? » (question ouverte n° 2 de la revue 2009, déjà restreinte aux états de Werner par Horodecki–Horodecki 1999) reste ouverte : les études numériques à 2 et 3 copies (Dür *et al.*, DiVincenzo *et al.*, 2000) n'ont pas fermé le trou.

Par contraste, le volet **PPT** de la question est réglé : PPT $\Rightarrow$ non distillable, car $\rho^{\Gamma} \geq 0$ entraîne $\langle\psi|(\rho^{\otimes n})^{\Gamma}|\psi\rangle \geq 0$ pour tout $|\psi\rangle$ et tout $n$ — les états à enchevêtrement lié (*bound entangled*) sont intriqués et définitivement non distillables.

## 6. Canaux « entanglement-breaking » : quand un canal détruit toute intrication

Un canal quantique est une application $\Phi$ complètement positive et préservant la trace. Horodecki, Shor et Ruskai (2003) définissent :

> **Définition 1.** $\Phi$ est *entanglement-breaking* (EB) si $(I \otimes \Phi)(\Gamma)$ est séparable pour tout état $\Gamma$ — tout état intriqué partagé avec un complice est détruit.

Leurs caractérisations calculables :

- **forme de Holevo** (mesure-puis-prépare) : $\Phi(\rho) = \sum_k R_k \operatorname{tr}(F_k \rho)$ avec $\{F_k\}$ une POVM et $R_k$ des états — le canal ne transmet que du classique ;
- **critère de Choi** : $\Phi$ est EB si et seulement si sa matrice de Choi $C_\Phi = (\Phi \otimes I)(|\Psi\rangle\langle\Psi|)$ (avec $|\Psi\rangle = \frac{1}{\sqrt d}\sum_i |ii\rangle$) est **séparable** ;
- structurellement, un canal d'un qubit est EB dès que l'image de la sphère de Bloch s'aplatit en un plan ou une droite.

L'**indice EB** (Lami–Giovannetti 2015) raffine : le $n$-indice d'un canal est le plus petit nombre d'applications **en série** qui rend le canal composé EB — les canaux EB ont indice 1, les canaux unitaires indice $\infty$. C'est le cadre de la question ouverte n° 38 de la revue 2009 (l'indice « PPT² » et ses variantes).

In [10]:
def choi_depolarisant(p, d):
    """Matrice de Choi C = (D_p (x) I)(|Psi><Psi|) du canal depolarisant
    D_p(rho) = (1-p) rho + p I/d, avec |Psi> = (1/sqrt(d)) somme_i |ii>."""
    C = np.zeros((d * d, d * d), dtype=complex)
    for a in range(d):
        for b in range(d):
            phi_ab = (1 - p) * np.outer(np.eye(d)[a], np.eye(d)[b])
            if a == b:
                phi_ab = phi_ab + p * np.eye(d) / d
            C += np.kron(phi_ab, np.outer(np.eye(d)[a], np.eye(d)[b])) / d
    return C


def ket_max_entrelace(d):
    return sum(ket(np.eye(d)[i], np.eye(d)[i]) for i in range(d)) / np.sqrt(d)


# Seuil EB du canal depolarisant (Lami-Giovannetti 2015, eq. 33) : EB ssi p >= d/(d+1).
# Le critere de Choi le traduit : C est un etat isotrope, separable ssi F = <Psi|C|Psi> <= 1/d.
print("d   p        F=<Psi|C|Psi>   vp(C^G_B) min   seuils : p >= d/(d+1) et F <= 1/d")
for d in [2, 3]:
    seuil_p = d / (d + 1)
    Psi = ket_max_entrelace(d)
    for p in [seuil_p - 0.05, seuil_p, seuil_p + 0.05]:
        C = choi_depolarisant(p, d)
        F = float(np.real(Psi.conj() @ (C @ Psi)))
        vp_min = val_propres(tp_b(C, d, d))[0]
        print(f"{d}   {p:.4f}   {F:.6f}        {vp_min:+.6f}"
              f"        ({seuil_p:.4f}, {1/d:.4f})")

d   p        F=<Psi|C|Psi>   vp(C^G_B) min   seuils : p >= d/(d+1) et F <= 1/d
2   0.6167   0.537500        -0.037500        (0.6667, 0.5000)
2   0.6667   0.500000        -0.000000        (0.6667, 0.5000)
2   0.7167   0.462500        +0.037500        (0.6667, 0.5000)
3   0.7000   0.377778        -0.022222        (0.7500, 0.3333)
3   0.7500   0.333333        -0.000000        (0.7500, 0.3333)
3   0.8000   0.288889        +0.022222        (0.7500, 0.3333)


### Lecture du résultat

La matrice de Choi du canal dépolarisant est un **état isotrope** de fidélité $F = \langle\Psi|C_\Phi|\Psi\rangle = (1-p) + p/d^2$. Aux deux dimensions testées, les deux seuils basculent **au même point** :

- $p < d/(d+1)$ : $F > 1/d$ et $C^{\Gamma_B}$ a une valeur propre négative — la Choi est intriquée, le canal **n'est pas EB** (il laisse passer de l'intrication) ;
- $p \geq d/(d+1)$ : $F \leq 1/d$ et la Choi devient PPT — pour un état isotrope, séparable $\iff$ PPT $\iff F \leq 1/d$ (revue 2009), donc le canal **est EB** : c'est la formule (33) de Lami–Giovannetti, $\Delta_\lambda \in \mathrm{EB} \iff \lambda \leq 1/(d+1)$ avec $\lambda = 1-p$.

On retrouve au passage la cohérence des sections : à $d=2$ le seuil $p \geq 2/3$ est exactement la frontière où la Choi — un état de la famille du §3 — cesse d'être NPT. Le canal dépolarisant **est** une porte sur le laboratoire Werner : bruit croissant = $p$ croissant = Choi migrant du régime intriqué vers le régime séparable.

In [11]:
# Indice EB du canal depolarisant (Lami-Giovannetti 2015, exemple 2) :
# la composition se fait sur lambda = 1-p par multiplication (D_p1 o D_p2 = D_{p1+p2-p1*p2}),
# donc D_p^n est EB ssi (1-p)^n <= 1/(d+1), et  n(D_p) = ceil( log(d+1) / log(1/(1-p)) ).
import math

print("d   p      n direct (plus petit n tel que (1-p)^n <= 1/(d+1))   n formule (34)")
for d in [2, 3]:
    seuil = 1 / (d + 1)
    for p in [0.5, 0.6, 0.65, 0.9]:
        n_direct = next(n for n in range(1, 200) if (1 - p) ** n <= seuil + 1e-12)
        n_formule = math.ceil(math.log(d + 1) / math.log(1 / (1 - p)))
        marque = "  <-- pas EB (p < d/(d+1))" if p < d / (d + 1) else ""
        print(f"{d}   {p:.2f}   n = {n_direct}                                      n = {n_formule}{marque}")

d   p      n direct (plus petit n tel que (1-p)^n <= 1/(d+1))   n formule (34)
2   0.50   n = 2                                      n = 2  <-- pas EB (p < d/(d+1))
2   0.60   n = 2                                      n = 2  <-- pas EB (p < d/(d+1))
2   0.65   n = 2                                      n = 2  <-- pas EB (p < d/(d+1))
2   0.90   n = 1                                      n = 1
3   0.50   n = 2                                      n = 2  <-- pas EB (p < d/(d+1))
3   0.60   n = 2                                      n = 2  <-- pas EB (p < d/(d+1))
3   0.65   n = 2                                      n = 2  <-- pas EB (p < d/(d+1))
3   0.90   n = 1                                      n = 1


### Lecture du résultat

Les deux colonnes coïncident en tout point : le plus petit nombre d'applications en série qui rend le canal dépolarisant EB suit exactement la formule $n(\Delta_\lambda) = \lceil \log(d+1) / \log(1/\lambda) \rceil$ (éq. 34 de Lami–Giovannetti). Lecture physique :

- un canal déjà EB ($p \geq d/(d+1)$) a $n = 1$ — une seule application détruit toute intrication ;
- en dessous, chaque application multiplie le paramètre de bruit résiduel $\lambda = 1-p$ par lui-même : l'indice compte combien de passages rapprochent $\lambda^n$ de $1/(d+1)$ — le canal est d'autant plus « dangereux » pour l'intrication que son indice est petit ;
- le canal identité ($p = 0$) aurait $n = \infty$, comme toute transformation unitaire.

L'indice EB et son cousin PPT (« $\Phi^2$ est-il EB dès que $\Phi$ est PPT ? ») donnent leur langage à la question ouverte n° 38 de la revue 2009 — le papier de Lami–Giovannetti cité en référence en calcule les valeurs explicites pour les grandes familles de canaux, dépolarisant en tête.

## 7. Exercices

Trois exercices pour s'approprier les organes du carnet (`werner_d`, la purification du §1, `choi_depolarisant`). Les stubs s'exécutent sans erreur ; à vous de les compléter.

### Exercice 1 — Werner en dimension 4

Le §3 a vérifié le seuil $\lambda = 1/2$ en $d = 3$. Refaites la vérification en $d = 4$ : pour $\lambda \in \{0.4, 0.5, 0.6\}$, calculez la valeur propre minimale de $W(\lambda)^{\Gamma_B}$ et constatez que la frontière PPT reste $\lambda = 1/2$ — la revue 2009 affirme que c'est le cas en **toute** dimension.

In [12]:
# Exercice a completer
# Etape 1 : construire W(lambda) pour d = 4 avec werner_d (section 3)
# Etape 2 : pour lambda dans [0.4, 0.5, 0.6], calculer val_propres(tp_b(W, 4, 4))[0]
# Etape 3 : verifier que la frontiere PPT est encore lambda = 1/2 et conclure
resultat_ex1 = None  # TODO etudiant : liste des couples (lambda, valeur propre minimale de W^G)

### Exercice 2 — purifier un état de Werner

Le §1 a purifié un qubit mixte. Purifiez maintenant $\rho = \text{werner\_2q}(0.5)$ : diagonalisez-le, construisez $|\Psi\rangle = \sum_k \sqrt{w_k}\,|a_k\rangle|k\rangle$ (système 4 dimensions × ancille 4 dimensions) et vérifiez que la trace partielle sur l'ancille redonne $\rho$ exactement. Le rang de Schmidt de votre purification vous surprend-il, au vu du §2 ?

In [13]:
# Exercice a completer
# Etape 1 : diagonaliser rho = werner_2q(0.5) via np.linalg.eigh (4 valeurs propres)
# Etape 2 : construire |Psi> = somme_k sqrt(w_k) |a_k>|k> avec ket() sur systeme x ancille (4 x 4)
# Etape 3 : verifier tr_ancille(|Psi><Psi|) == rho (reshape 4,4,4,4 puis einsum comme au §1)
# Indice : la purification d'un etat de rang 4 est generiquement de rang de Schmidt 4
resultat_ex2 = None  # TODO etudiant : le ket de purification |Psi>

### Exercice 3 — seuil EB du canal dépolarisant en dimension 3

Le §6 a localisé le seuil EB par trois points. Faites le balayage propre : pour $p$ de $0.6$ à $0.85$ par pas de $0.05$, calculez la valeur propre minimale de $C_\Phi^{\Gamma_B}$ pour le canal dépolarisant avec $d = 3$, repérez le changement de signe et comparez au seuil théorique $p \geq d/(d+1) = 3/4$.

In [14]:
# Exercice a completer
# Etape 1 : balayer p = 0.60, 0.65, ..., 0.85 avec choi_depolarisant(p, 3)
# Etape 2 : pour chaque p, afficher val_propres(tp_b(C, 3, 3))[0]
# Etape 3 : localiser le changement de signe et comparer a la frontiere p = 3/4
resultat_ex3 = None  # TODO etudiant : le seuil numerique observe

## À retenir

1. **Mixte ≠ superposition** : la pureté $\operatorname{tr}(\rho^2)$ sépare statistique classique (§1) et cohérence quantique ; toute matrice densité est l'état réduit d'un état pur plus grand (purification).
2. **Intrication d'un pur = spectre de l'état réduit** (Schmidt, §2) ; pour un mixte, il faut la décomposition en produits — indécidable par le seul spectre, d'où les critères.
3. **Werner $\lambda$** : une famille à un paramètre où séparable $\iff$ PPT $\iff \lambda \leq 1/2$ en toute dimension (§3), laboratoire de tout le reste.
4. **PPT** : $\rho^{\Gamma_B} \geq 0$ est nécessaire pour la séparabilité (Peres), suffisante en $2\times2$ et $2\times3$ (Horodecki *et al.*), définitivement insuffisante au-delà — enchevêtrement lié (§4).
5. **Distillabilité** : le critère de Schmidt rang 2 sépare NPT-distillable du trou $1/2 < \lambda \leq 3(d-1)/(2(2d-1))$ ; PPT $\Rightarrow$ non distillable ; « tout NPT est-il distillable ? » reste ouvert (§5, question ouverte n° 2).
6. **Canaux EB** : le canal casse toute intrication $\iff$ sa Choi est séparable $\iff$ il ne transmet que du classique (forme de Holevo) ; l'indice EB compte les applications en série pour y arriver (§6, question ouverte n° 38).

Le fil directeur : **chaque affirmation a été un nombre affiché par une cellule** — spectres, seuils, frontières numériques confondus avec les formules publiées, jamais seulement recopiées.

## Références

Toutes les publications sont lues et citées **depuis leur copie archivée** au gisement partagé `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\` (convention du dépôt : les PDF sous droits ne sont jamais committés) :

- A. Peres, *Separability Criterion for Density Matrices* (1996, arXiv 9604005) — `1996 - Peres - Separability Criterion for Density Matrices (arXiv 9604005).pdf` : critère PPT, éq. 3 et 7-8, exemples des §4.
- M. Horodecki, P. Horodecki, R. Horodecki, *Separability of Mixed States: Necessary and Sufficient Conditions* (1996, arXiv 9605038) — `1996 - Horodecki et al - Separability of Mixed States - Necessary and Sufficient Conditions (arXiv 9605038).pdf` : PPT décisif en $2\times2$ et $2\times3$.
- C. H. Bennett, G. Brassard, S. Popescu *et al.*, *Purification of Noisy Entanglement and Faithful Teleportation via Noisy Channels* (1996, arXiv 9511027) — `1996 - Bennett et al - Purification of Noisy Entanglement and Faithful Teleportation via Noisy Channels (arXiv 9511027).pdf` : la distillation.
- W. Dür, J. I. Cirac, M. Lewenstein, D. Bruß, *Distillability and Partial Transposition in Bipartite Systems* (2000, arXiv 9910022) — `2000 - Duer Cirac Lewenstein Bruss - Distillability and Partial Transposition in Bipartite Systems (arXiv 9910022).pdf` : seuil $3(d-1)/(2(2d-1))$, récursion $\lambda_N$ (§5).
- D. P. DiVincenzo, P. W. Shor, J. A. Smolin, B. M. Terhal, A. V. Thapliyal, *Evidence for Bound Entangled States with Negative Partial Transpose* (2000, arXiv 9910026) — `2000 - DiVincenzo et al - Evidence for Bound Entangled States with Negative Partial Transpose (arXiv 9910026).pdf` : le cadre numérique pseudo-$n$-copies (§5).
- M. Horodecki, P. W. Shor, M. B. Ruskai, *Entanglement Breaking Channels* (2003, arXiv 0302031) — `2003 - Horodecki Shor Ruskai - Entanglement Breaking Channels (arXiv 0302031).pdf` : définition EB, forme de Holevo, critère de Choi (§6).
- R. Horodecki, P. Horodecki, M. Horodecki, K. Horodecki, *Quantum Entanglement* (2009, arXiv 0702225) — `2009 - Horodecki et al - Quantum Entanglement (arXiv 0702225).pdf` : revue — états de Werner (éq. 71), critère de distillabilité (note 49), états isotropes, questions ouvertes n° 2 et 38.
- G. Lami, V. Giovannetti, *Entanglement-Breaking Indices* (2015, arXiv 1411.2517) — `2015 - Lami Giovannetti - Entanglement-Breaking Indices (arXiv 1411.2517).pdf` : seuil $\lambda \leq 1/(d+1)$ (éq. 33) et $n$-indice du dépolarisant (éq. 34, exemple 2).

L'état de Werner original (R. F. Werner, *Phys. Rev. A* **40**, 4277, 1989) n'a pas de préprint archivé au gisement ; il est cité ici via la paramétrisation de la revue 2009 (éq. 71), conformément à la règle de lecture avant attribution.